# DINOv2 Baseline Ablation Study Evaluation (Video-to-Video)

This notebook evaluates distance matrices generated by the ablation runs against the chosen **DINOv2 Baseline** on the DogReID-1553 closed-world video-to-video test set.

It computes and provides:
1. **mAP** (mean Average Precision) with 95% Bootstrap Confidence Intervals
2. **Rank-1, Rank-5, and Rank-10** Cumulative Matching Characteristic (CMC) with 95% Bootstrap Confidence Intervals
3. **Delta ($\Delta$) from Baseline** for Rank-1 and mAP to highlight relative gains/drops
4. **Categorized Breakdown Tables**:
   - **Backbone Adaptation Depth**: Frozen (0), 2, 4, 6 (Baseline), 8, 10, 12 blocks, and Full Fine-tuning
   - **Spatial & Multi-Layer Representation**: Pure `[CLS]` (Baseline), Patch Mean, Patch GeM, CLS+Mean, CLS+GeM, and Multi-Layer Fusion (`fuse4`)
   - **Feature Normalization & Neck**: BNNeck (Baseline) vs. No BNNeck (direct linear projection)
   - **Metric-Learning Objective**: Batch-Hard Triplet (Baseline), Soft Triplet, and Weighted Triplet
   - **Data Augmentation & Regularization**: RE + LS (Baseline), No RE, No LS, and AutoAugment
5. **Publication-Ready LaTeX Code Generator**: Pre-formatted tables directly pasteable into paper/appendix
6. **Visualizations**: Summary bar chart comparing all configurations against the baseline


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add root directory to sys.path
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR if (CURRENT_DIR / "evaluation").exists() else CURRENT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
if str(ROOT_DIR / "evaluation") not in sys.path:
    sys.path.append(str(ROOT_DIR / "evaluation"))

from evaluation.evaluation_utils import bootstrap_from_csv

print("Loaded evaluation_utils successfully!")
print(f"Project root: {ROOT_DIR}")


Loaded evaluation_utils successfully!
Project root: /d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553


In [2]:
# =====================================================================
# Configuration: Ablation Registry & Discovery
# =====================================================================
ABLATIONS_DIR = ROOT_DIR / "evaluation" / "csvs_ablations"

ABLATION_METADATA = {
    # Baseline
    "baseline": {
        "category": "0. Baseline Reference",
        "display_name": "DINOv2 Baseline (Chosen)",
        "config_desc": "Unfreeze 6, [CLS], BNNeck, Batch-Hard, RE (0.5), LS (0.1), Cosine",
        "order": 0
    },
    # 1. Depth
    "unfreeze0": {
        "category": "1. Backbone Adaptation Depth",
        "display_name": "Frozen Backbone (0 blocks)",
        "config_desc": "Only temporal pooling + BNNeck trained",
        "order": 10
    },
    "unfreeze2": {
        "category": "1. Backbone Adaptation Depth",
        "display_name": "Unfreeze 2 blocks",
        "config_desc": "Blocks 10-11 unfrozen",
        "order": 11
    },
    "unfreeze4": {
        "category": "1. Backbone Adaptation Depth",
        "display_name": "Unfreeze 4 blocks",
        "config_desc": "Blocks 8-11 unfrozen",
        "order": 12
    },
    "unfreeze6": {
        "category": "1. Backbone Adaptation Depth",
        "display_name": "Unfreeze 6 blocks (Baseline)",
        "config_desc": "Blocks 6-11 unfrozen",
        "order": 13
    },
    "unfreeze8": {
        "category": "1. Backbone Adaptation Depth",
        "display_name": "Unfreeze 8 blocks",
        "config_desc": "Blocks 4-11 unfrozen",
        "order": 14
    },
    "unfreeze10": {
        "category": "1. Backbone Adaptation Depth",
        "display_name": "Unfreeze 10 blocks",
        "config_desc": "Blocks 2-11 unfrozen",
        "order": 15
    },
    "unfreeze12": {
        "category": "1. Backbone Adaptation Depth",
        "display_name": "Unfreeze 12 blocks",
        "config_desc": "All 12 transformer blocks unfrozen",
        "order": 16
    },
    "full_finetune": {
        "category": "1. Backbone Adaptation Depth",
        "display_name": "Full Fine-tune",
        "config_desc": "All blocks + patch embed + final norm unfrozen",
        "order": 17
    },
    # 2. Token / Layer Representation
    "cls": {
        "category": "2. Spatial & Multi-Layer Representation",
        "display_name": "Pure [CLS] Token (Baseline)",
        "config_desc": "Final layer [CLS] token representation",
        "order": 20
    },
    "patch_mean": {
        "category": "2. Spatial & Multi-Layer Representation",
        "display_name": "Patch Mean Pooling",
        "config_desc": "Mean across spatial patch tokens",
        "order": 21
    },
    "patch_gem": {
        "category": "2. Spatial & Multi-Layer Representation",
        "display_name": "Patch GeM Pooling",
        "config_desc": "Generalized mean pooling (p=3.0) on patch tokens",
        "order": 22
    },
    "cls_mean": {
        "category": "2. Spatial & Multi-Layer Representation",
        "display_name": "CLS + Patch Mean",
        "config_desc": "Concatenation of CLS token and patch mean",
        "order": 23
    },
    "cls_gem": {
        "category": "2. Spatial & Multi-Layer Representation",
        "display_name": "CLS + Patch GeM",
        "config_desc": "Concatenation of CLS token and patch GeM",
        "order": 24
    },
    "fuse4": {
        "category": "2. Spatial & Multi-Layer Representation",
        "display_name": "Multi-Layer Fusion (fuse4)",
        "config_desc": "Softmax-weighted fusion across blocks 8-11",
        "order": 25
    },
    # 3. Normalization & Neck
    "bnneck": {
        "category": "3. Feature Normalization & Neck",
        "display_name": "BNNeck (Baseline)",
        "config_desc": "Linear projection + BatchNorm1d neck",
        "order": 30
    },
    "no_bnneck": {
        "category": "3. Feature Normalization & Neck",
        "display_name": "No BNNeck",
        "config_desc": "Direct linear projection without BatchNorm neck",
        "order": 31
    },
    # 4. Metric Loss
    "batch_hard": {
        "category": "4. Metric-Learning Objective",
        "display_name": "Batch-Hard Triplet (Baseline)",
        "config_desc": "Hard Euclidean triplet loss (margin = 0.3)",
        "order": 40
    },
    "soft_triplet": {
        "category": "4. Metric-Learning Objective",
        "display_name": "Soft Triplet Loss",
        "config_desc": "Smooth soft-margin triplet loss without fixed margin",
        "order": 41
    },
    "weighted_triplet": {
        "category": "4. Metric-Learning Objective",
        "display_name": "Weighted Triplet Loss",
        "config_desc": "Distance-weighted batch triplet loss",
        "order": 42
    },
    # 5. Regularization & Augmentation
    "standard_aug": {
        "category": "5. Data Augmentation & Regularization",
        "display_name": "RE + LS (Baseline)",
        "config_desc": "Random Erasing (p=0.5) + Label Smoothing (0.1)",
        "order": 50
    },
    "no_re": {
        "category": "5. Data Augmentation & Regularization",
        "display_name": "No Random Erasing",
        "config_desc": "Random Erasing probability p = 0.0",
        "order": 51
    },
    "no_ls": {
        "category": "5. Data Augmentation & Regularization",
        "display_name": "No Label Smoothing",
        "config_desc": "Standard cross-entropy (epsilon = 0.0)",
        "order": 52
    },
    "autoaug": {
        "category": "5. Data Augmentation & Regularization",
        "display_name": "AutoAugment (SBS)",
        "config_desc": "ImageNet AutoAugment policy (p=0.1)",
        "order": 53
    }
}

discovered_ablations = {}
if ABLATIONS_DIR.exists():
    for p in sorted(ABLATIONS_DIR.iterdir()):
        if not p.is_dir():
            continue
        key = p.name.replace("_vid2vid", "").replace("_img2vid", "")
        csv_candidates = list(p.glob("*closed*.csv")) + list(p.glob("*.csv"))
        for c in csv_candidates:
            if c.stat().st_size > 1000:
                meta = ABLATION_METADATA.get(key, {
                    "category": "6. Other Ablations",
                    "display_name": key.replace("_", " ").title(),
                    "config_desc": p.name,
                    "order": 99
                })
                discovered_ablations[key] = {
                    "key": key,
                    "folder": p.name,
                    "csv_path": c,
                    **meta
                }
                break

print(f"Total ablation directories discovered with valid CSVs: {len(discovered_ablations)}")
print("-" * 80)
for k, v in discovered_ablations.items():
    cat = v['category']
    disp = v['display_name']
    rel_path = v['csv_path'].relative_to(ROOT_DIR)
    print(f"[{cat}] {disp:<30} -> {rel_path}")


Total ablation directories discovered with valid CSVs: 4
--------------------------------------------------------------------------------
[0. Baseline Reference] DINOv2 Baseline (Chosen)       -> evaluation/csvs_ablations/baseline_vid2vid/closed_dist_matrix.csv
[2. Spatial & Multi-Layer Representation] Multi-Layer Fusion (fuse4)     -> evaluation/csvs_ablations/fuse4_vid2vid/closed_dist_matrix.csv
[3. Feature Normalization & Neck] No BNNeck                      -> evaluation/csvs_ablations/no_bnneck_vid2vid/closed_dist_matrix.csv
[1. Backbone Adaptation Depth] Frozen Backbone (0 blocks)     -> evaluation/csvs_ablations/unfreeze0_vid2vid/closed_dist_matrix.csv


In [3]:
# =====================================================================
# Run Bootstrap Evaluation on Available CSVs
# =====================================================================
M_ITERATIONS = 100  # Number of bootstrap resamples for 95% Confidence Intervals
all_ablation_results = {}

for key, item in discovered_ablations.items():
    csv_file = item["csv_path"]
    disp_name = item["display_name"]
    print("=" * 70)
    print(f"Evaluating {disp_name} ({key}) from {csv_file.name} ...")
    print("=" * 70)
    try:
        res = bootstrap_from_csv(str(csv_file), m=M_ITERATIONS, mode="closed")
        all_ablation_results[key] = {
            "meta": item,
            "metrics": res
        }
    except Exception as e:
        print(f"[ERROR] Failed to evaluate {key}: {e}")

print(f"\n[DONE] Successfully evaluated {len(all_ablation_results)} ablations!")


Evaluating DINOv2 Baseline (Chosen) (baseline) from closed_dist_matrix.csv ...
-> Calculating full set metrics (closed)...
-> Bootstrapping 'closed' metrics (100 iterations)...


100%|██████████| 100/100 [00:01<00:00, 55.62it/s]


Evaluating Multi-Layer Fusion (fuse4) (fuse4) from closed_dist_matrix.csv ...
-> Calculating full set metrics (closed)...
-> Bootstrapping 'closed' metrics (100 iterations)...


100%|██████████| 100/100 [00:01<00:00, 55.76it/s]


Evaluating No BNNeck (no_bnneck) from closed_dist_matrix.csv ...
-> Calculating full set metrics (closed)...
-> Bootstrapping 'closed' metrics (100 iterations)...


100%|██████████| 100/100 [00:01<00:00, 56.10it/s]


Evaluating Frozen Backbone (0 blocks) (unfreeze0) from closed_dist_matrix.csv ...
-> Calculating full set metrics (closed)...
-> Bootstrapping 'closed' metrics (100 iterations)...


100%|██████████| 100/100 [00:01<00:00, 54.42it/s]


[DONE] Successfully evaluated 4 ablations!


In [4]:
# =====================================================================
# Full Results Table (with Baseline comparison & deltas)
# =====================================================================
rows = []

# Baseline reference metrics for delta calculation
base_res = all_ablation_results.get("baseline", {}).get("metrics", {})
base_mAP = base_res.get("mAP_boot_mean", None)
base_cmc = base_res.get("cmc_boot_mean", None)
base_r1 = base_cmc[0] if base_cmc is not None and len(base_cmc) > 0 else None

for key, entry in all_ablation_results.items():
    meta = entry["meta"]
    res = entry["metrics"]
    full = res.get("full_set", {})
    
    # mAP
    m_mean = res.get("mAP_boot_mean", full.get("mAP", 0.0)) * 100
    m_ci = res.get("mAP_ci", (0.0, 0.0))
    m_low, m_high = m_ci[0] * 100, m_ci[1] * 100
    
    # CMC Rank-1, Rank-5, Rank-10
    cmc_boot = res.get("cmc_boot_mean", full.get("cmc", np.zeros(10)))
    cmc_l = res.get("cmc_ci_lower", cmc_boot)
    cmc_u = res.get("cmc_ci_upper", cmc_boot)
    
    r1 = cmc_boot[0] * 100
    r1_l, r1_u = cmc_l[0] * 100, cmc_u[0] * 100
    
    r5 = cmc_boot[4] * 100 if len(cmc_boot) >= 5 else 0.0
    r5_l, r5_u = cmc_l[4] * 100, cmc_u[4] * 100
    
    r10 = cmc_boot[9] * 100 if len(cmc_boot) >= 10 else 0.0
    r10_l, r10_u = cmc_l[9] * 100, cmc_u[9] * 100
    
    # Deltas
    delta_r1 = (r1 - base_r1 * 100) if base_r1 is not None else 0.0
    delta_map = (m_mean - base_mAP * 100) if base_mAP is not None else 0.0
    delta_str = f"{delta_r1:+.2f} / {delta_map:+.2f}" if key != "baseline" else "Reference"
    
    rows.append({
        "Key": key,
        "Order": meta.get("order", 99),
        "Category": meta.get("category", "Other"),
        "Variant": meta.get("display_name", key),
        "Description": meta.get("config_desc", ""),
        "Rank-1 (%) [95% CI]": f"{r1:.2f} [{r1_l:.2f}-{r1_u:.2f}]",
        "mAP (%) [95% CI]": f"{m_mean:.2f} [{m_low:.2f}-{m_high:.2f}]",
        "Rank-5 (%) [95% CI]": f"{r5:.2f} [{r5_l:.2f}-{r5_u:.2f}]",
        "Δ R-1 / Δ mAP": delta_str,
        "Rank-1_raw": r1,
        "mAP_raw": m_mean,
        "Rank-5_raw": r5
    })

df_all = pd.DataFrame(rows).sort_values("Order").reset_index(drop=True)

hdr_var = "Variant"
hdr_r1 = "Rank-1 [95% CI]"
hdr_map = "mAP [95% CI]"
hdr_delta = "Δ (R-1/mAP)"
hdr_cat = "Category"

print("=" * 120)
print(f"{hdr_var:<30} | {hdr_r1:<24} | {hdr_map:<24} | {hdr_delta:<14} | {hdr_cat}")
print("=" * 120)
for _, r in df_all.iterrows():
    v = r["Variant"]
    r1_s = r["Rank-1 (%) [95% CI]"]
    map_s = r["mAP (%) [95% CI]"]
    d_s = r["Δ R-1 / Δ mAP"]
    c_s = r["Category"]
    print(f"{v:<30} | {r1_s:<24} | {map_s:<24} | {d_s:<14} | {c_s}")
print("-" * 120)

display_cols = ["Category", "Variant", "Rank-1 (%) [95% CI]", "mAP (%) [95% CI]", "Rank-5 (%) [95% CI]", "Δ R-1 / Δ mAP", "Description"]
df_all[display_cols]


Variant                        | Rank-1 [95% CI]          | mAP [95% CI]             | Δ (R-1/mAP)    | Category
DINOv2 Baseline (Chosen)       | 61.28 [58.46-64.19]      | 57.63 [55.57-59.94]      | Reference      | 0. Baseline Reference
Frozen Backbone (0 blocks)     | 43.59 [40.69-46.27]      | 41.74 [39.55-43.94]      | -17.69 / -15.89 | 1. Backbone Adaptation Depth
Multi-Layer Fusion (fuse4)     | 55.74 [52.88-58.79]      | 53.04 [50.84-55.58]      | -5.55 / -4.59  | 2. Spatial & Multi-Layer Representation
No BNNeck                      | 59.90 [57.17-63.08]      | 56.44 [54.49-59.14]      | -1.38 / -1.19  | 3. Feature Normalization & Neck
------------------------------------------------------------------------------------------------------------------------


,Category,Variant,Rank-1 (%) [95% CI],mAP (%) [95% CI],Rank-5 (%) [95% CI],Δ R-1 / Δ mAP,Description
0,0. Baseline Reference,DINOv2 Baseline (Chosen),61.28 [58.46-64.19],57.63 [55.57-59.94],79.56 [77.21-81.74],Reference,"Unfreeze 6, [CLS], BNNeck, Batch-Hard, RE (0.5..."
1,1. Backbone Adaptation Depth,Frozen Backbone (0 blocks),43.59 [40.69-46.27],41.74 [39.55-43.94],65.97 [62.96-68.84],-17.69 / -15.89,Only temporal pooling + BNNeck trained
2,2. Spatial & Multi-Layer Representation,Multi-Layer Fusion (fuse4),55.74 [52.88-58.79],53.04 [50.84-55.58],76.39 [73.63-78.99],-5.55 / -4.59,Softmax-weighted fusion across blocks 8-11
3,3. Feature Normalization & Neck,No BNNeck,59.90 [57.17-63.08],56.44 [54.49-59.14],78.32 [75.94-80.56],-1.38 / -1.19,Direct linear projection without BatchNorm neck


In [5]:
# =====================================================================
# Breakdown Tables by Ablation Dimension
# =====================================================================
categories = sorted(df_all["Category"].unique())

for cat in categories:
    sub_df = df_all[df_all["Category"] == cat].copy()
    if cat != "0. Baseline Reference" and "baseline" in df_all["Key"].values:
        base_row = df_all[df_all["Key"] == "baseline"].copy()
        base_row["Variant"] = "[REF] DINOv2 Baseline"
        sub_df = pd.concat([base_row, sub_df], ignore_index=True)
        
    print(f"\n### {cat} ###")
    display_cols = ["Variant", "Rank-1 (%) [95% CI]", "mAP (%) [95% CI]", "Δ R-1 / Δ mAP", "Description"]
    print(sub_df[display_cols].to_string(index=False))



### 0. Baseline Reference ###
                 Variant Rank-1 (%) [95% CI]    mAP (%) [95% CI] Δ R-1 / Δ mAP                                                       Description
DINOv2 Baseline (Chosen) 61.28 [58.46-64.19] 57.63 [55.57-59.94]     Reference Unfreeze 6, [CLS], BNNeck, Batch-Hard, RE (0.5), LS (0.1), Cosine

### 1. Backbone Adaptation Depth ###
                   Variant Rank-1 (%) [95% CI]    mAP (%) [95% CI]   Δ R-1 / Δ mAP                                                       Description
     [REF] DINOv2 Baseline 61.28 [58.46-64.19] 57.63 [55.57-59.94]       Reference Unfreeze 6, [CLS], BNNeck, Batch-Hard, RE (0.5), LS (0.1), Cosine
Frozen Backbone (0 blocks) 43.59 [40.69-46.27] 41.74 [39.55-43.94] -17.69 / -15.89                            Only temporal pooling + BNNeck trained

### 2. Spatial & Multi-Layer Representation ###
                   Variant Rank-1 (%) [95% CI]    mAP (%) [95% CI] Δ R-1 / Δ mAP                                                       Descriptio